
# 練習問題: GPUで数値積分 (reduction)

## 目標

`target teams distribute parallel for` と `reduction(+:s)` を使って, 意味のある計算 (数値積分) を GPU 上で実行できるようになる. 総和に使うスカラ変数は `map` を書かなくても自動的に転送されることを確認する.

## 課題

中点則 (長方形近似) で次の積分を計算すると `π` になる.

```
∫_0^1 4/(1+x^2) dx = π
```

区間 `[0,1]` を `n` 等分し, 各小区間の中点 `x = (i + 0.5)/n` での値 `4/(1+x^2)` を足し合わせ, 幅 `dx = 1/n` を掛ける.

`gpu_integral.cpp` (または `gpu_integral.f90`) の総和ループが抜けている. `TODO` の箇所に, **GPU にオフロードして `reduction(+:s)` で総和 `s` を求めるループ** を書け.

- C++: `#pragma omp target teams distribute parallel for reduction(+:s)` とその直後の `for` ループ.
- Fortran: `!$omp target teams distribute parallel do reduction(+:s) private(x)` … `do` ループ … `!$omp end target teams distribute parallel do`.

考えどころ: `s` は総和を入れるスカラなので, 配列の `map` のような指定は不要 (スカラはコンパイラが自動で扱う). このため, データ移動 (`map`) を学ぶ前でもこの問題は解ける.

## コンパイルと実行

```
# C++
nvc++ -mp=gpu gpu_integral.cpp -o gpu_integral.exe

# Fortran
nvfortran -mp=gpu gpu_integral.f90 -o gpu_integral.exe
```

GPU は計算ノードにのみ搭載されているので `%%bash_submit` でジョブとして投入する.

```
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

./gpu_integral.exe 100000000
```

## 期待される結果

`n` が十分大きければ `pi` が `π ≈ 3.14159265...` に近づく.

```
pi  = 3.141592653...
error = ...e-...
```

- `OMP_NUM_TEAMS` を変えて (例: 32, 256, 1024) 結果が変わらないこと, 実行時間が変わることを確かめてみよ.
- `n` を大きくすると誤差が小さくなることも確認せよ.



# 1. AIチューター
- 以下は必要に応じて実行（毎度実行する必要はない）


In [ ]:
import heytutor


## 1-1. 一般的な質問
- ChatGPTなどに聞くときのように自由に質問可能。
- ただし「答えを教えて」などは自制すること。


In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 1-2. この問題に関するヒント
- `{file:problem.md}` は上記の問題文に展開される。


In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}


## 1-3. いくつかの変数
* それぞれ以下のように展開される。

* `{file:FILENAME}` : _FILENAME_ の中身
* `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前の入力・出力, etc.

## 1-4. 困ったときのヘルプ
* コンパイル時や実行時のエラー直後に以下を実行するとエラーに関するヘルプが得られる。


In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:gpu_integral.cpp}

コマンドと実行結果:
{bash[-1]}



## 1-5. フィードバック
* 答えが出た後も、無駄なところや、より良いやり方がないかを聞くことを推奨。
* 以下のファイル名は適宜書き換えよ (Fortranなら `.cpp` -> `.f90` とするなど)


In [ ]:
%%hey

フィードバックを下さい。

問題:
{file:problem.md}

私の答:
{file:gpu_integral.cpp}


# 2. ジョブ投入ツール
- 以下を実行しておくと、`%%bash_submit_a` (Aquariousへのジョブ投入), `%%bash_submit_o` (Odyssey へのジョブ投入) が使えるようになる


In [ ]:
import wisteria_submit

# 3. C++ ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ gpu_integral.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>

int main(int argc, char ** argv) {
  long n = (argc > 1 ? atol(argv[1]) : 100000000L);
  double dx = 1.0 / (double)n;
  double s = 0.0;

  /* 中点則で ∫_0^1 4/(1+x^2) dx = π を GPU 上で計算する.
     s は総和 (スカラ) なので reduction(+:s) を使う.
     スカラはコンパイラが自動的に転送するので map は不要. */
  // TODO: GPU上で reduction(+:s) を使って総和を求め π を計算せよ.

  double pi = s * dx;
  printf("n = %ld\n", n);
  printf("pi  = %.15f\n", pi);
  printf("M_PI = %.15f\n", M_PI);
  printf("error = %.3e\n", fabs(pi - M_PI));
  return 0;
}

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast gpu_integral.cpp -o gpu_integral_cpp.exe

## 3-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./gpu_integral_cpp.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a

./gpu_integral_cpp.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./gpu_integral_cpp.exe

## 3-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:gpu_integral.cpp}


# 4. Fortran ベースコード

In [ ]:
import heytutor

In [ ]:
%%writefile_ gpu_integral.f90
program gpu_integral
  character(len=32) :: arg
  integer(8) :: n, i
  real(8) :: dx, s, x, pi
  real(8), parameter :: pi_ref = 3.141592653589793d0
  n = 100000000_8
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  dx = 1.0d0 / dble(n)
  s = 0.0d0

  ! 中点則で ∫_0^1 4/(1+x^2) dx = π を GPU 上で計算する.
  ! s は総和 (スカラ) なので reduction(+:s) を使う.
  ! スカラはコンパイラが自動的に転送するので map は不要.
  ! TODO: GPU上で reduction(+:s) を使って総和を求め π を計算せよ.

  pi = s * dx
  print "(a,i0)", "n = ", n
  print "(a,f0.15)", "pi  = ", pi
  print "(a,f0.15)", "M_PI = ", pi_ref
  print "(a,es10.3)", "error = ", abs(pi - pi_ref)
end program gpu_integral

## 4-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast gpu_integral.f90 -o gpu_integral_f90.exe

## 4-2. Run
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./gpu_integral_f90.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a
./gpu_integral_f90.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit

#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./gpu_integral_f90.exe

## 4-3. 質問/フィードバック

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:gpu_integral.f90}